# Mapping the Potential Destructive Power of Wildfires Using Machine Learning
---
## Module 3: *Feature Engineering*
##### Version Number: 3.0
---
### Contents  
> 1. *Add Mean income per county*
> 2. *Calculate Temporal Fields*
> 3. *Calculate Lagged Weather Variables*
> 4. *Build Target*
> 5. *Fire History*
> 6. *File Export*
---
### Notes
> Features Added:
> - `Days_without_Rain` A rolling count that serves as a simple drought indicator
> - `Fire_History` An average count of fires per month in a region spanning the alst two years (needs refinement)
> - `Lagged_Variables` 7 day rolling averages for 'Avg Air Temp (F)', 'Precip (in)', 'Avg Rel Hum (%)', 'Avg Wind Speed (mph)'
---
### Inputs
- `trimmed.csv` cleaned weather data joined with cleaned fire damage dataset
---
### Outputs 
- `final.csv` - final clean dataset with features added
---
### User Defined Dependencies

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

# Show basic facts about a dataframe
from src.data_utils import basic_explore

# basic health checks after a merge
from src.data_utils import post_merge_check

---
### Third Party Dependencies

In [2]:
# Core data tools
import pandas as pd
import numpy as np
from datetime import datetime

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

---

## Load Datasets

In [3]:
samples = pd.read_csv("../data/processed/samples_projected.csv")
mean_income = pd.read_csv("../data/raw/mean_income_by_county.csv")

## 1. Add Mean income per county

In [4]:
basic_explore(mean_income)

Rows:  58  Columns:  2
Duplicates  0
Total NA values:  0  of  116 datapoints


In [5]:
#mean_income['County'] = mean_income['County'].str.upper()
mean_income['Mean Income'] = (
    mean_income['Mean Income']
    .str.replace(',', '', regex=False)  # Remove commas
    .astype(int)                        # Convert to integer
)

mean_income.head(2)

,County,Mean Income
0,Alameda,138489
1,Alpine,107737


In [6]:
samples_income = samples.merge(
    mean_income,
    left_on='County', right_on='County',
    how='left', indicator=True
)

In [7]:
post_merge_check(samples_income,samples)

Premerged shape:  (446340, 42)
Merged shape:  (446340, 44)
Duplicates before merge:  0
Duplicates after merge:  0
NA values before merge:  0
NA values after merge:  0


In [8]:
samples_income.head(2)

,Sample_ID,Date,Burning Index,Energy Release Component,Actual Evapotranspiration,100-hour Dead Fuel Moisture,1000-hour Dead Fuel Moisture,Precipitation,Maximum Relative Humidity,Minimum Relative Humidity,...,Eco_ID,Buffered_Area_Km,Population_Density,Housing_Density,geometry,fire_count,total_fire_damage,acres,Mean Income,_merge
0,1,2018-01-01,24.0,37.0,1.7,16.6,13.7,0.0,97.800003,52.700001,...,1,2376.745850,1028.087158,388.035278,POINT (259634.34221583148 -577730.614588792),0,0.0,0.0,111241,both
1,72,2018-01-01,27.0,49.0,1.5,11.5,13.2,0.0,63.799999,18.100000,...,2,4070.743408,69.062691,21.709528,POINT (24042.27612160902 -193099.7829722697),0,0.0,0.0,75449,both


---

## 2. Calculate Temporal Fields
`Winter` = 0, `Spring` = 1, `Summer` = 2, `Fall` = 3

In [9]:
def get_season(date):
    month = date.month
    if month in [12, 1, 2]:
        return 0
    elif month in [3, 4, 5]:
        return 1
    elif month in [6, 7, 8]:
        return 2
    else:
        return 3
    
## Apply function   
samples_income['Date'] = pd.to_datetime(samples_income['Date'])
samples_income['Season'] = samples_income['Date'].apply(get_season)
samples_income['Month'] = samples_income['Date'].dt.month
samples_income['Year'] = samples_income['Date'].dt.year

## 3. Calculate Lagged Weather Variables

We calculate 7-day rolling averages for select weather variables to capture recent trends that may influence fire severity.

In [10]:
# Sort data by County and Date to prepare for rolling average
samples_income = samples_income.sort_values(by=['Sample_ID', 'Date'])

# Define columns for 7-day rolling average
avg_columns = [
    "Vapor Pressure Deficit",
    "Precipitation",
    "Solar Radiation",
    "Daily Maximum Air Temperature",
    "Daily Minimum Air Temperature",
    "Maximum Relative Humidity",
    "Minimum Relative Humidity",
    "Wind Speed"
]

# Apply rolling mean by County
for col in avg_columns:
    samples_income[f'{col} 7 Day Avg'] = samples_income.groupby('Sample_ID')[col].rolling(window=7, min_periods=1).mean().reset_index(level=0, drop=True)

## 4. Build Final Target

Create a categorical target that represents the risk levels of damage caused.\
\
`Low` Risk = 0, No fires started on the day\
`Medium` Risk = 1, Fires present but limited property damage done, under 1 million dollars total\
`High` Risk = 2, Fires causing over 1 million dollars in damage 

In [11]:
def fire_risk_category(row):
    # Low risk: no fires
    if (row['total_fire_damage'] > 0) or (row['acres'] > 0):
        return 2

    # Moderate risk: at least one fire AND damage under 1 million
    elif row['fire_count'] > 0:
        return 1

    # High risk: all other cases
    else:
        return 0

## Apply function
samples_income['Target'] = samples_income.apply(fire_risk_category, axis=1)

## Display resulting risk category assignments
samples_income['Target'].value_counts()

Target
0    389137
1     43278
2     13925
Name: count, dtype: int64

## 5.  Historical fires per month

Create a new column that records the historical average number of fires for each month, based on all previous years up to that point. Each row should reflect the average number of fires that occurred in the same calendar month across prior years, providing a historical baseline for comparison.

**ASSUMPTION** -Since the weather coverage in this dataset is limited, no previous year readings exists.
For now, I will just use the current count as a rough estimation for the average. 

In [12]:
# Convert Datetime column to string
samples_income['Month_Year'] = samples_income['Date'].dt.strftime('%Y-%m')
month_year = samples_income['Month_Year'].unique()

In [13]:
# Dictionary to store month-to-count mapping
monthly_fire_counts = {}

# Step 1: Calculate fire counts for each month
for month in month_year:
    # Get index and subset
    month_index = samples_income[samples_income['Month_Year'] == month].index
    subset = samples_income.loc[month_index]
    
    # Eliminate 'Low' days
    total = subset[subset['Target'] != 0]
    
    # Store count
    monthly_fire_counts[month] = len(total)

# Step 2: Calculate 2-year rolling average
# Convert month_year to sorted list to ensure order
month_year_sorted = sorted(month_year)

# Dictionary for rolling averages
rolling_avg = {}

for i, month in enumerate(month_year_sorted):
    if i == 0:
        # First year: average of all *future* years (exclude current)
        future_values = list(monthly_fire_counts.values())[1:]
        rolling_avg[month] = np.mean(future_values) if future_values else 0

    elif i == 1:
        # Second year: average of first year and current
        prev = month_year_sorted[0]
        rolling_avg[month] = np.mean([monthly_fire_counts[prev], monthly_fire_counts[month]])
    
    else:
        # Rolling average of previous 2 years
        prev1 = month_year_sorted[i - 1]
        prev2 = month_year_sorted[i - 2]
        rolling_avg[month] = np.mean([monthly_fire_counts[prev1], monthly_fire_counts[prev2]])

# Step 3: Assign back to each row in the dataframe
for month in month_year_sorted:
    month_index = samples_income[samples_income['Month_Year'] == month].index
    samples_income.loc[month_index, '2-Year Avg Fires'] = rolling_avg[month]

In [14]:
drop = ['Month_Year']
samples_income = samples_income.drop(columns=drop)

In [15]:
samples_income

,Sample_ID,Date,Burning Index,Energy Release Component,Actual Evapotranspiration,100-hour Dead Fuel Moisture,1000-hour Dead Fuel Moisture,Precipitation,Maximum Relative Humidity,Minimum Relative Humidity,...,Vapor Pressure Deficit 7 Day Avg,Precipitation 7 Day Avg,Solar Radiation 7 Day Avg,Daily Maximum Air Temperature 7 Day Avg,Daily Minimum Air Temperature 7 Day Avg,Maximum Relative Humidity 7 Day Avg,Minimum Relative Humidity 7 Day Avg,Wind Speed 7 Day Avg,Target,2-Year Avg Fires
0,1,2018-01-01,24.0,37.0,1.7,16.600000,13.700000,0.0,97.800003,52.700001,...,0.410000,0.0,145.600006,290.700012,283.000000,97.800003,52.700001,1.300000,0,679.178571
343,1,2018-01-02,25.0,39.0,2.0,15.900000,13.800000,0.0,90.599998,43.200001,...,0.545000,0.0,138.900002,292.550003,284.149994,94.200001,47.950001,1.250000,0,679.178571
364,1,2018-01-03,23.0,36.0,1.6,15.900000,13.800000,0.0,93.500000,53.400002,...,0.530000,0.0,115.900002,292.666667,284.766663,93.966667,49.766668,1.233333,0,679.178571
678,1,2018-01-04,27.0,38.0,2.3,16.299999,14.100000,0.0,98.300003,47.700001,...,0.537500,0.0,124.725000,293.050003,284.824997,95.050001,49.250001,1.400000,0,679.178571
856,1,2018-01-05,31.0,38.0,3.1,16.000000,14.200000,0.0,89.699997,46.299999,...,0.564000,0.0,130.100002,293.360004,285.159998,93.980000,48.660001,1.720000,0,679.178571
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
445554,173,2025-01-19,23.0,28.0,1.7,12.600000,20.700001,0.0,76.199997,28.600000,...,0.374286,0.0,116.600001,278.614284,265.642857,73.342857,30.028571,2.342857,0,602.000000
445710,173,2025-01-20,26.0,29.0,1.8,12.600000,20.200001,0.0,77.000000,26.700001,...,0.380000,0.0,118.142857,278.542855,265.414285,70.971428,28.657143,2.528571,0,602.000000
445894,173,2025-01-21,27.0,32.0,2.4,11.300000,19.700001,0.0,53.599998,20.200001,...,0.370000,0.0,119.885715,278.099997,265.142857,69.871428,28.414286,2.685714,0,602.000000
446120,173,2025-01-22,26.0,34.0,2.3,10.300000,19.299999,0.0,51.900002,18.799999,...,0.360000,0.0,121.742858,277.714281,264.585711,69.257143,27.800000,2.628571,0,602.000000


---

## 6. Export Data

In [16]:
samples_income.to_csv("../data/processed/samples_income.csv", index=False)
print("All datasets saved successfully to ../data/processed/")

All datasets saved successfully to ../data/processed/
